In [9]:
import numpy as np
import pandas as pd
import pickle, time

In [4]:
PATH = "data/train_triplets.txt"

In [10]:
user_map, song_map = {}, {}

In [11]:
def encode(series, mapping):
    for v in series.unique():
        if v not in mapping:
            mapping[v] = len(mapping)
    return series.map(mapping).to_numpy(dtype="int32")

In [12]:
users, songs, plays = [], [], []
t0 = time.time()

In [13]:
reader = pd.read_csv(
    PATH, sep="\t", header=None,
    names=["user", "song", "plays"],
    dtype={"plays": "int32"},
    chunksize=2_000_000,
)

In [14]:
for i, chunk in enumerate(reader):
    users.append(encode(chunk["user"], user_map))
    songs.append(encode(chunk["song"], song_map))
    plays.append(chunk["plays"].to_numpy())
    print(i, len(user_map), len(song_map), f"{time.time()-t0:.0f}s")

0 41756 193683 29s
1 83296 243366 33s
2 125473 272507 37s
3 167503 292756 41s
4 209542 307782 46s
5 251250 319181 52s
6 293819 328343 58s
7 336203 336003 63s
8 378236 342643 67s
9 420709 348436 72s
10 463348 353045 76s
11 506293 357088 80s
12 548203 360725 84s
13 589716 363995 88s
14 631532 367043 92s
15 673097 369736 97s
16 715295 372103 101s
17 757748 374357 105s
18 799793 376274 109s
19 841936 378120 113s
20 884353 379815 118s
21 926734 381386 122s
22 969268 382847 126s
23 1011475 384300 130s
24 1019318 384546 132s


In [16]:
users = np.concatenate(users)
songs = np.concatenate(songs)
plays = np.concatenate(plays)
print(users.shape, songs.shape, plays.shape)

(48373586,) (48373586,) (48373586,)


In [17]:
np.savez("data/triplets_int.npz", user=users, song=songs, plays=plays)
with open("data/id_maps.pkl", "wb") as f:
    pickle.dump((list(user_map), list(song_map)), f)

In [18]:
d = np.load("data/triplets_int.npz")
users, songs, plays = d["user"], d["song"], d["plays"]
print(users.shape, songs.shape, plays.shape)

(48373586,) (48373586,) (48373586,)


In [19]:
print("min plays:", plays.min())
print("max plays:", plays.max())
print("mean plays:", plays.mean())
print("median plays:", np.median(plays))
print("share of rows with plays == 1:", (plays == 1).mean())

min plays: 1
max plays: 9667
mean plays: 2.866858847305635
median plays: 1.0
share of rows with plays == 1: 0.5944559495754563


In [20]:
qs = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
print(dict(zip(qs, np.quantile(plays, qs))))

for t in [2, 5, 10, 50, 100, 1000]:
    print(f"plays >= {t}: {(plays >= t).sum():>10,}  ({(plays >= t).mean():.4%})")

{0.5: np.float64(1.0), 0.75: np.float64(3.0), 0.9: np.float64(6.0), 0.95: np.float64(10.0), 0.99: np.float64(24.0), 0.999: np.float64(69.0)}
plays >= 2: 19,617,620  (40.5544%)
plays >= 5:  7,261,443  (15.0112%)
plays >= 10:  2,477,559  (5.1217%)
plays >= 50:    107,042  (0.2213%)
plays >= 100:     19,317  (0.0399%)
plays >= 1000:         63  (0.0001%)


In [22]:
songs_per_user = np.bincount(users)   # number of rows per user -> distinct songs
n_users, n_songs = len(songs_per_user), songs.max() + 1

qs = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
print("min/max/mean:", songs_per_user.min(), songs_per_user.max(), songs_per_user.mean())
print(dict(zip(qs, np.quantile(songs_per_user, qs))))

for t in [5, 10, 20, 50]:
    print(f"users with < {t} songs: {(songs_per_user < t).mean():.2%}")
    
print("density:", len(users) / (int(n_users) * int(n_songs)))

min/max/mean: 10 4400 47.45681524313316
{0.01: np.float64(10.0), 0.1: np.float64(12.0), 0.25: np.float64(16.0), 0.5: np.float64(27.0), 0.75: np.float64(55.0), 0.9: np.float64(105.0), 0.99: np.float64(284.0)}
users with < 5 songs: 0.00%
users with < 10 songs: 0.00%
users with < 20 songs: 35.14%
users with < 50 songs: 71.64%
density: 0.00012340998279304207


In [23]:
users_per_song = np.bincount(songs)   # number of distinct listeners per song

print("min/max/mean:", users_per_song.min(), users_per_song.max(), users_per_song.mean())
print(dict(zip(qs, np.quantile(users_per_song, qs))))

for t in [2, 5, 10, 50]:
    print(f"songs with < {t} listeners: {(users_per_song < t).mean():.2%}")

top = np.sort(users_per_song)[::-1]
print("share of all rows held by top 100 songs:", top[:100].sum() / len(songs))
print("share of all rows held by top 1% of songs:", top[:len(top)//100].sum() / len(songs))

min/max/mean: 1 110479 125.79401684063805
{0.01: np.float64(1.0), 0.1: np.float64(2.0), 0.25: np.float64(4.0), 0.5: np.float64(13.0), 0.75: np.float64(52.0), 0.9: np.float64(218.0), 0.99: np.float64(1970.0)}
songs with < 2 listeners: 8.26%
songs with < 5 listeners: 25.87%
songs with < 10 listeners: 42.47%
songs with < 50 listeners: 74.39%
share of all rows held by top 100 songs: 0.07093553080807365
share of all rows held by top 1% of songs: 0.3777230408347233


In [24]:
for t in [2, 5, 10, 20, 50]:
    keep = users_per_song >= t
    print(f"min {t:>2} listeners: songs kept {keep.sum():>7,} ({keep.mean():.1%}), "
          f"rows kept {keep[songs].mean():.1%}")

min  2 listeners: songs kept 352,765 (91.7%), rows kept 99.9%
min  5 listeners: songs kept 285,061 (74.1%), rows kept 99.5%
min 10 listeners: songs kept 221,216 (57.5%), rows kept 98.6%
min 20 listeners: songs kept 161,173 (41.9%), rows kept 96.9%
min 50 listeners: songs kept  98,485 (25.6%), rows kept 92.9%


In [25]:
for t in [10, 20, 50]:
    row_mask = (users_per_song >= t)[songs]
    spu = np.bincount(users[row_mask], minlength=int(n_users))
    print(f"min {t:>2} listeners: users with 0 songs {(spu == 0).sum():>6,}, "
          f"< 10 songs {(spu < 10).mean():.1%}, median songs {np.median(spu):.0f}")

min 10 listeners: users with 0 songs      5, < 10 songs 0.6%, median songs 27
min 20 listeners: users with 0 songs     31, < 10 songs 1.4%, median songs 27
min 50 listeners: users with 0 songs    143, < 10 songs 3.5%, median songs 25


In [26]:
total_plays = np.bincount(users, weights=plays)   # total plays per user

top = np.argsort(plays)[-10:][::-1]               # 10 rows with the highest play counts
for i in top:
    u, s = users[i], songs[i]
    print(f"plays {plays[i]:>5}  song listeners {users_per_song[s]:>6}  "
          f"user songs {songs_per_user[u]:>4}  user total plays {total_plays[u]:>8.0f}")

plays  9667  song listeners   4345  user songs  202  user total plays    13132
plays  3534  song listeners   1369  user songs  183  user total plays     4370
plays  3532  song listeners  84000  user songs  122  user total plays     3758
plays  2948  song listeners    499  user songs   26  user total plays     3151
plays  2381  song listeners    276  user songs   26  user total plays     2658
plays  2368  song listeners    152  user songs   64  user total plays     2834
plays  2213  song listeners   1847  user songs  325  user total plays     3657
plays  2165  song listeners    757  user songs  150  user total plays     5381
plays  2130  song listeners    205  user songs   16  user total plays     2212
plays  1961  song listeners    470  user songs   10  user total plays     2219


In [27]:
n_users, n_songs = int(users.max()) + 1, int(songs.max()) + 1

def filter_iter(min_song, min_user):
    mask = np.ones(len(users), dtype=bool)
    it = 0
    while True:
        song_cnt = np.bincount(songs[mask], minlength=n_songs)
        user_cnt = np.bincount(users[mask], minlength=n_users)
        new = mask & (song_cnt >= min_song)[songs] & (user_cnt >= min_user)[users]
        it += 1
        if new.sum() == mask.sum():
            return new, it
        mask = new

for min_song, min_user in [(20, 10), (50, 10), (20, 20)]:
    m, it = filter_iter(min_song, min_user)
    print(f"songs>={min_song}, users>={min_user}: iters {it}, "
          f"rows {m.sum():,} ({m.mean():.1%}), "
          f"users {len(np.unique(users[m])):,}, songs {len(np.unique(songs[m])):,}")

songs>=20, users>=10: iters 9, rows 46,750,539 (96.6%), users 1,004,635, songs 160,261
songs>=50, users>=10: iters 9, rows 44,588,508 (92.2%), users 983,094, songs 97,436
songs>=20, users>=20: iters 8, rows 41,618,741 (86.0%), users 645,822, songs 151,944


In [29]:
mask, it = filter_iter(100, 10)

old_users, new_users = np.unique(users[mask], return_inverse=True)
old_songs, new_songs = np.unique(songs[mask], return_inverse=True)

u = new_users.astype("int32")
s = new_songs.astype("int32")
p = plays[mask]

print("rows:", len(u))
print("users:", len(old_users), " max new user id + 1:", u.max() + 1)
print("songs:", len(old_songs), " max new song id + 1:", s.max() + 1)

rows: 42038519
users: 956770  max new user id + 1: 956770
songs: 64519  max new song id + 1: 64519


In [30]:
np.savez("data/filtered_100_10.npz",
         user=u, song=s, plays=p,
         old_user=old_users, old_song=old_songs)

## Testing with the filtered data

In [31]:
d = np.load("data/filtered_100_10.npz")
u, s, p = d["user"], d["song"], d["plays"]

print("rows:", len(u))
print("users:", u.max() + 1, " songs:", s.max() + 1)
print("min songs per user:", np.bincount(u).min())
print("min listeners per song:", np.bincount(s).min())

rows: 42038519
users: 956770  songs: 64519
min songs per user: 10
min listeners per song: 100
